In [30]:
import os, glob, json, random
import numpy as np
import h5py
from tqdm import tqdm

# ===== 路径配置 =====
IN_DIR  = r"E:/rf_datasets_IQ_raw"        # 原始 72 个 .mat 文件目录
OUT_DIR = r"E:/rf_datasets_IQ_eq"         # 等化后输出目录（会创建）
REF_MAT = r"lte_ref/ideal_dmrs_256.mat"   # MATLAB 导出的理想参考（len=256）

os.makedirs(OUT_DIR, exist_ok=True)

# ===== DMRS 结构配置 =====
NFFT_HALF = 128         # 每段长度（去CP后的一个SC-FDMA符号）
TOTAL_LEN = 256         # 两段拼接
OCC_LEN   = 72          # 6 RB = 72 occupied subcarriers（固定窗口长度）

# ===== 等化超参数（先用默认）=====
THR_RX = 0.15           # 接收侧有效bin门控阈值（相对每条样本该段最大幅度）
GUARD  = 2              # 噪声估计时占用带两侧额外排除的 guard bins

# ===== HDF5 输出压缩 =====
COMPRESS_LEVEL = 4

# ===== 调试开关 =====
MAX_FILES = 0           # 0=处理全部；>0=只处理前 MAX_FILES 个
MAX_ROWS_PER_FILE = 0   # 0=处理文件内全部样本；>0=只处理前 MAX_ROWS_PER_FILE 条

In [31]:
def read_compound_complex(arr):
    """
    支持 MATLAB v7.3 的 compound(real/imag) 或直接 complex
    """
    if isinstance(arr, np.ndarray) and arr.dtype.fields and ("real" in arr.dtype.fields) and ("imag" in arr.dtype.fields):
        return arr["real"] + 1j * arr["imag"]
    if np.iscomplexobj(arr):
        return arr
    return np.array(arr)

def load_dmrs_Nx256(mat_path: str) -> np.ndarray:
    """
    返回 dmrs: shape (N, 256) complex64
    兼容 (256, N) 自动转置
    """
    with h5py.File(mat_path, "r") as f:
        rf = f["rfDataset"]
        dmrs_struct = rf["dmrs"][:]  # compound real/imag
        dmrs = read_compound_complex(dmrs_struct).astype(np.complex64)

    if dmrs.ndim != 2:
        raise ValueError(f"dmrs must be 2D, got {dmrs.shape}")

    if dmrs.shape[1] == TOTAL_LEN:
        return dmrs
    if dmrs.shape[0] == TOTAL_LEN:
        return dmrs.T
    raise ValueError(f"dmrs shape unexpected: {dmrs.shape} (expected (N,256) or (256,N))")

def write_eq_mat(in_fp: str, out_fp: str, dmrs_eq: np.ndarray, compress_level=4):
    """
    复制原 rfDataset 到新文件，并新增 dmrs_eq（compound real/imag）
    dmrs_eq: shape (N,256) complex
    """
    dmrs_eq = dmrs_eq.astype(np.complex64)
    dt = np.dtype([("real", np.float32), ("imag", np.float32)])
    out_arr = np.empty(dmrs_eq.shape, dtype=dt)
    out_arr["real"] = dmrs_eq.real.astype(np.float32)
    out_arr["imag"] = dmrs_eq.imag.astype(np.float32)

    with h5py.File(in_fp, "r") as fin, h5py.File(out_fp, "w") as fout:
        fin.copy("rfDataset", fout)
        rf_out = fout["rfDataset"]
        if "dmrs_eq" in rf_out:
            del rf_out["dmrs_eq"]
        rf_out.create_dataset(
            "dmrs_eq", data=out_arr,
            compression="gzip", compression_opts=compress_level
        )

In [32]:
def load_ideal_dmrs_256(ref_mat_path: str) -> np.ndarray:
    with h5py.File(ref_mat_path, "r") as f:
        x = f["ideal_dmrs_256"][()]
        ideal = read_compound_complex(x).squeeze()
    ideal = np.array(ideal).astype(np.complex64)
    if ideal.ndim != 1 or ideal.shape[0] != TOTAL_LEN:
        raise ValueError(f"ideal_dmrs_256 must be shape (256,), got {ideal.shape}")
    ideal = ideal / (np.sqrt(np.mean(np.abs(ideal)**2)) + 1e-12)
    return ideal

def fftshift_fft_rows(x: np.ndarray) -> np.ndarray:
    """
    x: (N, nfft) complex
    return: (N, nfft) fftshift(fft(x))
    """
    X = np.fft.fft(x, axis=1)
    X = np.fft.fftshift(X, axes=1)
    return X

ideal = load_ideal_dmrs_256(REF_MAT)
ideal0 = ideal[:NFFT_HALF]
ideal1 = ideal[NFFT_HALF:TOTAL_LEN]

S0 = fftshift_fft_rows(ideal0.reshape(1, -1))[0]
S1 = fftshift_fft_rows(ideal1.reshape(1, -1))[0]

print("Loaded ideal reference OK. len=", ideal.shape[0])

Loaded ideal reference OK. len= 256


In [33]:
def best_contiguous_band(P: np.ndarray, occ_len: int):
    """
    P: (nfft,) nonnegative power
    return: start index, occ indices, energy_ratio
    """
    nfft = P.shape[0]
    best_s = 0
    best_val = -1.0
    for s in range(0, nfft - occ_len + 1):
        val = float(np.sum(P[s:s+occ_len]))
        if val > best_val:
            best_val = val
            best_s = s
    occ = np.arange(best_s, best_s + occ_len, dtype=np.int64)
    energy_ratio = best_val / (float(np.sum(P)) + 1e-12)
    return best_s, occ, energy_ratio

P0 = np.abs(S0)**2
P1 = np.abs(S1)**2

s0, occ0, er0 = best_contiguous_band(P0, OCC_LEN)
s1, occ1, er1 = best_contiguous_band(P1, OCC_LEN)

print(f"seg0: occ={OCC_LEN}/{NFFT_HALF}, range=[{occ0[0]},{occ0[-1]}], energy_ratio={er0:.3f}")
print(f"seg1: occ={OCC_LEN}/{NFFT_HALF}, range=[{occ1[0]},{occ1[-1]}], energy_ratio={er1:.3f}")

# 若 energy_ratio 很低（比如 <0.7），通常表示 ideal 参考与真实提取段不一致（符号切片/窗口/模式等）

seg0: occ=72/128, range=[30,101], energy_ratio=0.967
seg1: occ=72/128, range=[30,101], energy_ratio=0.967


In [34]:
def mmse_equalize_half_batch(x_half: np.ndarray,
                            S: np.ndarray,
                            occ: np.ndarray,
                            thr_rx: float = 0.20,
                            guard: int = 2) -> np.ndarray:
    """
    x_half: (N,128) complex
    S:      (128,)  complex reference FFTshift(FFT(ideal))
    occ:    (72,)   occupied bin indices (fftshift domain)
    return: (N,128) complex equalized time-domain symbol (cp-free)
    """
    eps = 1e-12
    x = x_half.astype(np.complex64)

    # per-row unit power
    p = np.mean(np.abs(x)**2, axis=1, keepdims=True) + eps
    x = x / np.sqrt(p)

    # freq: (N,128)
    X = fftshift_fft_rows(x)

    # noise variance from bins outside occupied window (with guard)
    nfft = X.shape[1]
    mask_null = np.ones(nfft, dtype=bool)
    mask_null[occ] = False
    s, e = int(occ[0]), int(occ[-1])
    mask_null[max(0, s-guard):min(nfft, e+guard+1)] = False
    null = X[:, mask_null]
    sigma2 = np.median(np.abs(null)**2, axis=1, keepdims=True) if null.size else np.median(np.abs(X)**2, axis=1, keepdims=True)

    # work only on occupied bins
    Y = X[:, occ]  # (N,72)
    absY = np.abs(Y)
    mx = absY.max(axis=1, keepdims=True) + eps
    maskY = absY > (thr_rx * mx)

    S_occ = S[occ].reshape(1, -1)

    # LS channel estimate + one-tap MMSE
    H = Y / (S_occ + eps)
    W = np.conj(H) / (np.abs(H)**2 + sigma2 + eps)
    Yeq = np.where(maskY, W * Y, 0.0).astype(np.complex64)

    # rebuild full spectrum
    Xeq = np.zeros_like(X)
    Xeq[:, occ] = Yeq

    # back to time
    xeq = np.fft.ifft(np.fft.ifftshift(Xeq, axes=1), axis=1)

    # re-normalize
    peq = np.mean(np.abs(xeq)**2, axis=1, keepdims=True) + eps
    xeq = xeq / np.sqrt(peq)

    return xeq.astype(np.complex64)

In [35]:
mat_files = sorted(glob.glob(os.path.join(IN_DIR, "*.mat")))
if MAX_FILES and MAX_FILES > 0:
    mat_files = mat_files[:MAX_FILES]

print("Found mat files:", len(mat_files))

# 随机挑一个文件，小跑 200 条看看是否能跑通
tmp_fp = random.choice(mat_files)
dmrs = load_dmrs_Nx256(tmp_fp)
print("Sample file:", os.path.basename(tmp_fp), "dmrs shape:", dmrs.shape)

take = min(200, dmrs.shape[0])
x = dmrs[:take, :]

x0 = x[:, :NFFT_HALF]
x1 = x[:, NFFT_HALF:TOTAL_LEN]

x0_eq = mmse_equalize_half_batch(x0, S0, occ0, thr_rx=THR_RX, guard=GUARD)
x1_eq = mmse_equalize_half_batch(x1, S1, occ1, thr_rx=THR_RX, guard=GUARD)
x_eq = np.concatenate([x0_eq, x1_eq], axis=1)

print("eq shape:", x_eq.shape)
print("orig power median:", float(np.median(np.mean(np.abs(x)**2, axis=1))))
print("eq   power median:", float(np.median(np.mean(np.abs(x_eq)**2, axis=1))))

Found mat files: 72
Sample file: rfDataset_TX002_RX004_loc01_20kmh_20260125_013640.mat dmrs shape: (2924, 256)
eq shape: (200, 256)
orig power median: 0.00017616976401768625
eq   power median: 1.0


In [36]:
meta = {
    "method": "WiSig-style equalization: LS channel estimation + one-tap MMSE (per occupied bin), applied on 2x128 cp-free DMRS symbols then concatenated",
    "nfft_half": NFFT_HALF,
    "total_len": TOTAL_LEN,
    "occ_len": OCC_LEN,
    "thr_rx": THR_RX,
    "guard": GUARD,
    "occ0_range": [int(occ0[0]), int(occ0[-1])],
    "occ1_range": [int(occ1[0]), int(occ1[-1])],
    "occ0_energy_ratio": float(er0),
    "occ1_energy_ratio": float(er1),
    "ref_mat": os.path.abspath(REF_MAT),
}
with open(os.path.join(OUT_DIR, "equalization_meta.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

for fp in tqdm(mat_files, desc="Equalizing LTE-V mats (2x128, fixed 72-bin window)"):
    dmrs = load_dmrs_Nx256(fp)  # (N,256)
    if MAX_ROWS_PER_FILE and MAX_ROWS_PER_FILE > 0:
        dmrs = dmrs[:MAX_ROWS_PER_FILE, :]

    if dmrs.shape[1] != TOTAL_LEN:
        raise ValueError(f"{os.path.basename(fp)}: expected 256 columns, got {dmrs.shape}")

    x0 = dmrs[:, :NFFT_HALF]          # (N,128)
    x1 = dmrs[:, NFFT_HALF:TOTAL_LEN] # (N,128)

    x0_eq = mmse_equalize_half_batch(x0, S0, occ0, thr_rx=THR_RX, guard=GUARD)
    x1_eq = mmse_equalize_half_batch(x1, S1, occ1, thr_rx=THR_RX, guard=GUARD)

    dmrs_eq = np.concatenate([x0_eq, x1_eq], axis=1)  # (N,256)

    out_fp = os.path.join(OUT_DIR, os.path.basename(fp))
    write_eq_mat(fp, out_fp, dmrs_eq, compress_level=COMPRESS_LEVEL)

print("Done. Equalized mats saved to:", OUT_DIR)

Equalizing LTE-V mats (2x128, fixed 72-bin window): 100%|██████████| 72/72 [00:14<00:00,  4.83it/s]

Done. Equalized mats saved to: E:/rf_datasets_IQ_eq


In [37]:
test_in = random.choice(mat_files)
test_out = os.path.join(OUT_DIR, os.path.basename(test_in))

orig = load_dmrs_Nx256(test_in)
with h5py.File(test_out, "r") as f:
    rf = f["rfDataset"]
    keys = list(rf.keys())
    dmrs_eq_struct = rf["dmrs_eq"][:]
    dmrs_eq = read_compound_complex(dmrs_eq_struct).astype(np.complex64)

print("File:", os.path.basename(test_in))
print("rfDataset keys:", keys)
print("orig shape:", orig.shape, "eq shape:", dmrs_eq.shape)
print("orig power median:", float(np.median(np.mean(np.abs(orig)**2, axis=1))))
print("eq   power median:", float(np.median(np.mean(np.abs(dmrs_eq)**2, axis=1))))

File: rfDataset_TX004_RX002_loc01_20kmh_20260125_013832.mat
rfDataset keys: ['NCellID', 'Nfft', 'dmrs', 'dmrsOFDMSymbols', 'dmrs_eq', 'frameLength', 'fs', 'fs_proc', 'fs_raw', 'locID', 'note', 'numFrames', 'rxID', 'sourceFile', 'txID', 'velocity']
orig shape: (2999, 256) eq shape: (2999, 256)
orig power median: 2.3771954147377983e-06
eq   power median: 1.0


In [38]:
import numpy as np, os, random, glob, h5py
from tqdm import tqdm

def load_dmrs_eq_Nx256(mat_path: str) -> np.ndarray:
    with h5py.File(mat_path, "r") as f:
        rf = f["rfDataset"]
        dmrs_eq_struct = rf["dmrs_eq"][:]
        dmrs_eq = read_compound_complex(dmrs_eq_struct).astype(np.complex64)
    if dmrs_eq.shape[1] == 256:
        return dmrs_eq
    if dmrs_eq.shape[0] == 256:
        return dmrs_eq.T
    raise ValueError(dmrs_eq.shape)

def corr_to_ref(x, ref):
    # x/ref: (N,128) complex
    num = np.abs(np.sum(np.conj(ref.reshape(1,-1)) * x, axis=1))
    den = (np.linalg.norm(ref) * np.linalg.norm(x, axis=1) + 1e-12)
    return num / den

out_files = sorted(glob.glob(os.path.join(OUT_DIR, "*.mat")))
sample_files = random.sample(out_files, k=min(6, len(out_files)))

corr0_before, corr1_before = [], []
corr0_after,  corr1_after  = [], []

for fp_out in sample_files:
    fp_in = os.path.join(IN_DIR, os.path.basename(fp_out))
    x_in  = load_dmrs_Nx256(fp_in)
    x_eq  = load_dmrs_eq_Nx256(fp_out)

    take = min(800, x_in.shape[0])
    idx = np.random.choice(x_in.shape[0], size=take, replace=False)

    in0 = x_in[idx, :128]
    in1 = x_in[idx, 128:256]
    eq0 = x_eq[idx, :128]
    eq1 = x_eq[idx, 128:256]

    # 单位功率归一（避免幅度影响相关）
    in0 = in0 / (np.sqrt(np.mean(np.abs(in0)**2, axis=1, keepdims=True)) + 1e-12)
    in1 = in1 / (np.sqrt(np.mean(np.abs(in1)**2, axis=1, keepdims=True)) + 1e-12)
    eq0 = eq0 / (np.sqrt(np.mean(np.abs(eq0)**2, axis=1, keepdims=True)) + 1e-12)
    eq1 = eq1 / (np.sqrt(np.mean(np.abs(eq1)**2, axis=1, keepdims=True)) + 1e-12)

    corr0_before.extend(corr_to_ref(in0, ideal0))
    corr1_before.extend(corr_to_ref(in1, ideal1))
    corr0_after.extend(corr_to_ref(eq0, ideal0))
    corr1_after.extend(corr_to_ref(eq1, ideal1))

print("Corr(seg0) before: median=%.3f  after: median=%.3f" % (np.median(corr0_before), np.median(corr0_after)))
print("Corr(seg1) before: median=%.3f  after: median=%.3f" % (np.median(corr1_before), np.median(corr1_after)))

# 门控过强会出现等化后相关很低、甚至接近0的尾部；看一下 5% 分位数
print("Corr(seg0) after p05=%.3f" % (np.quantile(corr0_after, 0.05)))
print("Corr(seg1) after p05=%.3f" % (np.quantile(corr1_after, 0.05)))

Corr(seg0) before: median=0.080  after: median=0.282
Corr(seg1) before: median=0.075  after: median=0.272
Corr(seg0) after p05=0.027
Corr(seg1) after p05=0.027
